In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))
import inventory as I
import manifest as M

In [ ]:
(open_picu_dir/"derived").mkdir(exist_ok=True)
(open_picu_dir/"reports").mkdir(exist_ok=True)

# SMH

In [ ]:
smh_dir = Path("/home/jormond/Documents/DS217/users/jormond/SMH")
smh_m, smh_w = M.smh_manifest(smh_dir)
M.print_waterfall(smh_w, "SMH")
smh_m.to_csv(open_picu_dir/"derived"/"SMH_manifest.csv", index=False)

# RBH

In [ ]:
rbh_dir = Path("/home/jormond/Documents/DS217/users/jormond/RBH")
rbh_m, rbh_w = M.rbh_manifest(rbh_dir)
M.print_waterfall(rbh_w, "RBH")
rbh_m.to_csv(open_picu_dir/"derived"/"RBH_manifest.csv", index=False)

# GOSH

In [ ]:
gosh_dir = Path("/home/jormond/Documents/DS217/users/jormond/GOSH")
gosh_m, gosh_w = M.gosh_manifest(gosh_dir)
M.print_waterfall(gosh_w, "GOSH")
gosh_m.to_csv(open_picu_dir/"derived"/"GOSH_manifest.csv", index=False)

# Years

In [ ]:
rbh_years = M.describe(rbh_m, "PATIENT_RK")
smh_years = M.describe(smh_m, "UniquePtID")
gosh_years = M.describe(gosh_m, "project_id")
pd.concat({"RBH": rbh_years["stays"], "SMH": smh_years["stays"], "GOSH": gosh_years["stays"]}, axis=1)

### RBH - 2013 and beyond stays not in picanet

In [ ]:
dem = pd.read_csv(rbh_dir/"ai_picu_demographics.csv", usecols=["studyID", "PICUadm_dttm"]).drop_duplicates("studyID")
dem["stay_key"] = M.rbh_stay_key(dem["studyID"])
dem["year"] = pd.to_datetime(dem["PICUadm_dttm"], format=M.RBH_TIME_FMT).dt.year
dropped = dem[(dem["year"] >= 2013) & ~dem["stay_key"].isin(set(rbh_m["stay_key"]))].drop_duplicates("stay_key")
print(dropped["year"].value_counts().sort_index().to_string())

## Baseline: manifest waterfalls (before the step 1 and step 2 rebuild)
Reruns the three manifests with the current code and saves each waterfall as a CSV in `derived/reports/baseline/`. After the rebuild we rerun this cell and compare; any difference is a regression. Counts only, no IDs.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import manifest as M

base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
out = open_picu_dir / "derived" / "reports" / "baseline"
out.mkdir(parents=True, exist_ok=True)

rbh_m, rbh_w = M.rbh_manifest(base / "RBH")
smh_m, smh_w = M.smh_manifest(base / "SMH")
gosh_m, gosh_w = M.gosh_manifest(base / "GOSH")

for site, w, m in [("RBH", rbh_w, rbh_m), ("SMH", smh_w, smh_m), ("GOSH", gosh_w, gosh_m)]:
    M.print_waterfall(w, site)
    pd.DataFrame(w).to_csv(out / f"{site}_waterfall.csv", index=False)
    print(site, "manifest stays:", len(m), "\n")